In [2]:
import pandas as pd
import json
import numpy as np
import altair as alt

prices = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/pp434/pp434_semi_anonymised_prices.parquet', engine='fastparquet')
items = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/pp434/pp434_semi_anonymised_items.parquet', engine='fastparquet')

In [15]:
prices.head()

,store_id,product_id,date,price,unit_price,loyalty_price,original_price
163,1,293258696,2024-02-01,11.90,£2.38/kg,NaN,NaN
370,1,313519686,2024-01-24,1.95,£2.05/100g,NaN,NaN
416,5,4088600288949,2023-08-20,0.52,£0.05 / 100g,NaN,NaN
617,8,343891725517256,2025-08-27,2.60,£1.45/100g,NaN,NaN
631,3,910000033641,2023-08-08,2.25,£1.42/100g,NaN,NaN


In [16]:
items.head()

,store_id,product_id,segment_code,description
0,3,5532146,CP0116901,CANNED FRUIT
1,1,315462322,CP0111311,"CAKES, TARTS AND SWEET PIES"
2,5,4061461116484,CP0118602,"ICE CREAM BARS, LOLLIES AND CONES"
3,3,1000383146075,CP0116601,FROZEN FRUIT
4,3,910000540889,CP0116601,FROZEN FRUIT


In [17]:
merged_df = pd.merge(prices, items, on=['store_id', 'product_id'], how='inner')
merged_df.head()

,store_id,product_id,date,price,unit_price,loyalty_price,original_price,segment_code,description
0,1,293258696,2024-02-01,11.90,£2.38/kg,NaN,NaN,CP0111101,"RICE, IN ALL FORMS (EXCL. RICE FLOUR)"
1,1,313519686,2024-01-24,1.95,£2.05/100g,NaN,NaN,CP0111101,"RICE, IN ALL FORMS (EXCL. RICE FLOUR)"
2,5,4088600288949,2023-08-20,0.52,£0.05 / 100g,NaN,NaN,CP0111101,"RICE, IN ALL FORMS (EXCL. RICE FLOUR)"
3,8,343891725517256,2025-08-27,2.60,£1.45/100g,NaN,NaN,CP0111101,"RICE, IN ALL FORMS (EXCL. RICE FLOUR)"
4,3,910000033641,2023-08-08,2.25,£1.42/100g,NaN,NaN,CP0111101,"RICE, IN ALL FORMS (EXCL. RICE FLOUR)"


In [18]:
merged_df.description.unique()

array(['RICE, IN ALL FORMS (EXCL. RICE FLOUR)', 'FLOUR, WHEAT-BASED',
       'BREAD, WHITE', 'BREAD, BROWN OR SEEDED',
       'BREAD ROLLS, BUNS, BAGUETTES AND OTHER LOAVES',
       'FLATBREADS, THINS AND PITTAS',
       'BREAD SIDE DISHES (E.G. GARLIC BREAD)',
       'OTHER BREAKFAST BAKERY PRODUCTS', 'BISCUITS, SWEET',
       'BISCUITS, SAVOURY', 'CAKES, TARTS AND SWEET PIES',
       'BREAKFAST CEREALS', 'CEREAL BARS AND CEREAL-BASED SNACKS',
       'OATS AND PORRIDGE', 'PASTA AND NOODLES, DRY OR FRESH',
       'PASTA AND NOODLES, PACKET OR POT', 'COUSCOUS',
       'MEAT OF COWS, FRESH, CHILLED OR FROZEN',
       'MEAT OF PIGS, FRESH, CHILLED OR FROZEN',
       'MEAT OF GOATS, LAMBS AND SHEEP, FRESH, CHILLED OR FROZEN',
       'MEAT OF CHICKEN, FRESH, CHILLED OR FROZEN',
       'COOKED HAM AND CONTINENTAL MEATS (E.G. SALAMI)',
       'COOKED POULTRY, SLICES AND DELI FOODS',
       'PORK, DRIED, SALTED OR SMOKED',
       'SAUSAGES AND SIMILAR MEAT PRODUCTS',
       'BREADED CHICKEN AN

In [19]:
merged_df.store_id.unique()

array([1, 5, 8, 3, 2, 4, 9])

In [20]:
merged_df.dtypes

store_id                   int64
product_id                 int64
date              datetime64[ns]
price                    float64
unit_price                object
loyalty_price            float64
original_price           float64
segment_code              object
description               object
dtype: object

In [21]:
# Filter for wine-related descriptions
wine_descriptions = merged_df[merged_df['description'].str.contains('wine', case=False, na=False)]['description'].unique()
print(f"Found {len(wine_descriptions)} wine-related product descriptions:\n")
for desc in sorted(wine_descriptions):
    print(f"- {desc}")

Found 5 wine-related product descriptions:

- WINE, CHAMPAGNE AND SPARKLING
- WINE, FORTIFIED (INCL. SHERRY AND PORT)
- WINE, RED
- WINE, ROSE
- WINE, WHITE


In [22]:
# Create a dataframe with only red, rose, and white wine
wine_df = merged_df[merged_df['description'].isin(['WINE, RED', 'WINE, ROSE', 'WINE, WHITE'])].copy()
print(f"Wine dataframe shape: {wine_df.shape}")
wine_df.head()

Wine dataframe shape: (215391, 9)


,store_id,product_id,date,price,unit_price,loyalty_price,original_price,segment_code,description
4076657,8,1787986668667,2024-10-21,7.49,£7.49/75cl,NaN,8.99,CP0212101,"WINE, WHITE"
4076658,8,849374306943070,2025-09-01,10.00,£10/75cl,NaN,NaN,CP0212101,"WINE, WHITE"
4076659,8,1201559405941,2024-10-21,11.99,£11.99/75cl,NaN,NaN,CP0212101,"WINE, WHITE"
4076660,2,7937216,2025-11-10,8.75,£8.75 / 75cl,7.0,NaN,CP0212101,"WINE, WHITE"
4076661,8,5012157083370834,2024-10-21,6.99,£6.99/75cl,NaN,NaN,CP0212101,"WINE, WHITE"


In [23]:
# Prepare aggregated data for histogram
# Create price bins with 0.5 increments starting at 0
bin_edges = np.arange(0, 24.5, 0.5)  # 0.0, 0.5, 1.0, ..., 24.0
wine_df['price_bin'] = pd.cut(wine_df['price'], bins=bin_edges, include_lowest=True)
wine_df['price_bin_center'] = wine_df['price_bin'].apply(lambda x: x.mid if pd.notna(x) else None)

# Aggregate counts by store, description, and price bin
agg_df = wine_df.groupby(['store_id', 'description', 'price_bin_center']).size().reset_index(name='count')

# Add "All Stores" aggregation
all_stores_agg = wine_df.groupby(['description', 'price_bin_center']).size().reset_index(name='count')
all_stores_agg['store_id'] = 0  # Use 0 to represent "All Stores"

# Combine both datasets
combined_agg = pd.concat([agg_df, all_stores_agg], ignore_index=True)

# Get list of stores for dropdown
stores = sorted([s for s in wine_df['store_id'].unique().tolist()])
store_options = [0] + stores  # 0 represents "All Stores"
store_labels = ['All Stores'] + [f'Store {s}' for s in stores]

# Create store selection dropdown
store_selection = alt.binding_select(options=store_options, labels=store_labels, name='Select Store: ')
store_param = alt.param(name='selected_store', value=0, bind=store_selection)

# Create wine type selection dropdown
wine_types = ['All Types', 'WINE, RED', 'WINE, ROSE', 'WINE, WHITE']
wine_type_selection = alt.binding_select(options=wine_types, name='Select Wine Type: ')
wine_type_param = alt.param(name='selected_wine_type', value='All Types', bind=wine_type_selection)

# Define colors for each wine type
color_scale = alt.Scale(
    domain=['WINE, RED', 'WINE, ROSE', 'WINE, WHITE'],
    range=['#8B0000', '#FFB6C1', '#FFD700']  # Dark red, light pink, gold
)

C:\Users\juanx\AppData\Local\Temp\ipykernel_7396\2668825712.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_df = wine_df.groupby(['store_id', 'description', 'price_bin_center']).size().reset_index(name='count')
C:\Users\juanx\AppData\Local\Temp\ipykernel_7396\2668825712.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  all_stores_agg = wine_df.groupby(['description', 'price_bin_center']).size().reset_index(name='count')


In [24]:
# Pre-calculate mean, median, and box plot statistics for all combinations
stats_list = []

# 1. Stats per store per wine type
for store in stores:
    for wine_type in ['WINE, RED', 'WINE, ROSE', 'WINE, WHITE']:
        subset = wine_df[(wine_df['store_id'] == store) & (wine_df['description'] == wine_type)]
        if len(subset) > 0:
            stats_list.append({
                'store_id': store,
                'description': wine_type,
                'mean_price': subset['price'].mean(),
                'median_price': subset['price'].median(),
                'q1_price': subset['price'].quantile(0.25),
                'q3_price': subset['price'].quantile(0.75),
                'min_price': subset['price'].min(),
                'max_price': subset['price'].max()
            })

# 2. Stats per store (all types combined)
for store in stores:
    subset = wine_df[wine_df['store_id'] == store]
    if len(subset) > 0:
        stats_list.append({
            'store_id': store,
            'description': 'All Types',
            'mean_price': subset['price'].mean(),
            'median_price': subset['price'].median(),
            'q1_price': subset['price'].quantile(0.25),
            'q3_price': subset['price'].quantile(0.75),
            'min_price': subset['price'].min(),
            'max_price': subset['price'].max()
        })

# 3. Stats per wine type (all stores combined)
for wine_type in ['WINE, RED', 'WINE, ROSE', 'WINE, WHITE']:
    subset = wine_df[wine_df['description'] == wine_type]
    if len(subset) > 0:
        stats_list.append({
            'store_id': 0,  # 0 represents "All Stores"
            'description': wine_type,
            'mean_price': subset['price'].mean(),
            'median_price': subset['price'].median(),
            'q1_price': subset['price'].quantile(0.25),
            'q3_price': subset['price'].quantile(0.75),
            'min_price': subset['price'].min(),
            'max_price': subset['price'].max()
        })

# 4. Overall stats (all stores, all types)
stats_list.append({
    'store_id': 0,
    'description': 'All Types',
    'mean_price': wine_df['price'].mean(),
    'median_price': wine_df['price'].median(),
    'q1_price': wine_df['price'].quantile(0.25),
    'q3_price': wine_df['price'].quantile(0.75),
    'min_price': wine_df['price'].min(),
    'max_price': wine_df['price'].max()
})

stats_df = pd.DataFrame(stats_list)
print(f"Statistics calculated for {len(stats_df)} combinations")
stats_df.head(10)

Statistics calculated for 32 combinations


,store_id,description,mean_price,median_price,q1_price,q3_price,min_price,max_price
0,1,"WINE, RED",9.823107,9.00,7.50,11.75,2.75,22.0
1,1,"WINE, ROSE",8.633164,8.50,5.75,10.50,2.50,20.0
2,1,"WINE, WHITE",8.665633,8.25,6.50,10.00,2.50,20.0
3,2,"WINE, RED",9.815538,9.00,7.25,12.00,2.75,22.0
4,2,"WINE, ROSE",8.344709,8.00,5.65,10.25,2.35,20.0
5,2,"WINE, WHITE",9.153193,8.50,7.00,10.75,2.50,20.0
6,3,"WINE, RED",8.396212,7.75,6.47,9.50,2.75,21.5
7,3,"WINE, ROSE",7.589108,7.00,5.45,9.00,2.48,20.0
8,3,"WINE, WHITE",7.965710,7.25,5.75,9.00,2.50,20.0
9,4,"WINE, RED",9.228915,9.00,7.00,11.00,2.75,22.0


In [33]:
# Create the histogram with stacked bars
histogram = alt.Chart(combined_agg).mark_bar(
    opacity=0.7,
    strokeWidth=0,
    size=9  # Width of bars in pixels
).encode(
    alt.X('price_bin_center:Q', title='Price (£)', scale=alt.Scale(domain=[0, 24], nice=False)),
    alt.Y('count:Q', stack=True, title='Count'),
    alt.Color('description:N', 
        scale=color_scale, 
        title='Wine Type'
    ),
    order=alt.Order('description:N', sort='ascending'),  # Ascending order: RED bottom, ROSE middle, WHITE top
    tooltip=[
        alt.Tooltip('description:N', title='Wine Type'),
        alt.Tooltip('price_bin_center:Q', title='Price', format='.2f'),
        alt.Tooltip('count:Q', title='Count')
    ]
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    (wine_type_param == 'All Types') | (alt.datum.description == wine_type_param)
).add_params(
    store_param,
    wine_type_param
).properties(
    width=700,
    height=400,
    title='Wine Price Distribution by Type'
)

# Create box plot layer
# Box (Q1 to Q3)
box_rect = alt.Chart(stats_df).mark_bar(
    size=30,
    opacity=0.6,
    color='navy'
).encode(
    x=alt.X('q1_price:Q', scale=alt.Scale(domain=[0, 24]), title=None),
    x2='q3_price:Q',
    y=alt.value(20),  # Fixed y position at bottom of chart
    tooltip=[
        alt.Tooltip('store_id:N', title='Store'),
        alt.Tooltip('description:N', title='Wine Type'),
        alt.Tooltip('min_price:Q', title='Min', format='.2f'),
        alt.Tooltip('q1_price:Q', title='Q1', format='.2f'),
        alt.Tooltip('median_price:Q', title='Median', format='.2f'),
        alt.Tooltip('q3_price:Q', title='Q3', format='.2f'),
        alt.Tooltip('max_price:Q', title='Max', format='.2f')
    ]
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

# Whiskers (min to Q1 and Q3 to max)
whisker_min = alt.Chart(stats_df).mark_rule(
    color='navy',
    strokeWidth=2
).encode(
    x='min_price:Q',
    x2='q1_price:Q',
    y=alt.value(20)
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

whisker_max = alt.Chart(stats_df).mark_rule(
    color='navy',
    strokeWidth=2
).encode(
    x='q3_price:Q',
    x2='max_price:Q',
    y=alt.value(20)
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

# End caps for whiskers
cap_min = alt.Chart(stats_df).mark_tick(
    color='navy',
    thickness=2,
    size=15
).encode(
    x='min_price:Q',
    y=alt.value(20)
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

cap_max = alt.Chart(stats_df).mark_tick(
    color='navy',
    thickness=2,
    size=15
).encode(
    x='max_price:Q',
    y=alt.value(20)
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

# Median line on box
median_mark = alt.Chart(stats_df).mark_tick(
    color='black',
    thickness=3,
    size=30
).encode(
    x='median_price:Q',
    y=alt.value(20)
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

# Mean marker on box (diamond shape)
mean_mark = alt.Chart(stats_df).mark_point(
    shape='diamond',
    color='red',
    size=100,
    filled=True
).encode(
    x='mean_price:Q',
    y=alt.value(20),
    tooltip=[
        alt.Tooltip('mean_price:Q', title='Mean Price', format='.2f')
    ]
).transform_filter(
    alt.datum.store_id == store_param
).transform_filter(
    alt.datum.description == wine_type_param
)

# Combine all layers
final_chart = (histogram + box_rect + whisker_min + whisker_max + cap_min + cap_max + median_mark + mean_mark).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_legend(
    orient='right'
)

final_chart

alt.LayerChart(...)

In [34]:
# Save the visualization as JSON
final_chart.save('../graphs/autocpi_wine_price_histogram.json')
print("Chart saved to graphs/autocpi_wine_price_histogram.json")

Chart saved to graphs/autocpi_wine_price_histogram.json
